# Workshop: Delta Lake Optimization — From Fragmentation to Performance

Practice end-to-end Delta Lake optimization: compaction, time travel, cloning, and clustering.

**Prerequisites:** 03 — Optimization Demo

<!-- TRAINER-BOX -->

> **TRAINER INSTRUCTIONS**
>
> | Tier | Target audience | Time | Goal |
> |---|---|---|---|
> | **Guided** | New to Delta Lake | 45 min | Follow examples, fill in blanks |
> | **Independent** | 1+ year Databricks | 30 min | Read acceptance criteria only, write from scratch |
>
> - Ensure cluster is running before participants start the Setup cell
> - All participants must complete Setup before any task
> - Task 6 requires the Spark UI — remind participants to open the SQL tab before running benchmark queries

<!-- LAB-SCENARIO -->

## Scenario

> *"An e-commerce platform stores orders in Delta Lake. The ETL pipeline appends small batches — one file per micro-batch — which has caused severe file fragmentation over time. Your task is to diagnose the problem, compact and clean the table, understand the change history, protect the data through cloning, and finally benchmark three optimization strategies to determine which one delivers the best query performance for product-level filtering."*

<!-- LAB-OBJECTIVES -->

## Learning Objectives

After completing this lab you will be able to:

- Create and inspect a fragmented Delta table using `DESCRIBE DETAIL` and `DESCRIBE HISTORY`
- Compact files with `OPTIMIZE` and remove obsolete data with `VACUUM`
- Apply DML operations (`INSERT`, `UPDATE`, `DELETE`) and audit them via the transaction log
- Query row-level change history using **Change Data Feed** (`table_changes`)
- Query and restore historical table versions using Delta Time Travel
- Create isolated table copies with `DEEP CLONE`
- Compare data-skipping effectiveness of Z-ORDER vs Liquid Clustering vs an unoptimized baseline

## Setup

In [0]:
%run ../../setup/00_setup

In [0]:
# Define table names used throughout this workshop
TABLE_BASE   = f"{CATALOG}.{BRONZE_SCHEMA}.orders_lab"
TABLE_ZORDER = f"{CATALOG}.{BRONZE_SCHEMA}.orders_lab_zorder"
TABLE_LIQUID = f"{CATALOG}.{BRONZE_SCHEMA}.orders_lab_liquid"

print(f"Base table    : {TABLE_BASE}")
print(f"Z-ORDER table : {TABLE_ZORDER}")
print(f"Liquid table  : {TABLE_LIQUID}")

## Preparation — Creating a Fragmented Delta Table

In production, ETL pipelines often write one small file per batch — especially with streaming or
frequent micro-batch jobs. This creates hundreds of tiny Parquet files, which is the primary cause
of slow Delta scan performance.

The setup below **generates 2 million synthetic orders** spread across 300 small files, which
simulates a table that has received months of micro-batch writes. This scale is necessary to make
the Z-ORDER vs Liquid Clustering performance difference measurable.

**Change Data Feed (CDF)** is also enabled on the table. CDF records every row-level change
(INSERT, UPDATE, DELETE) in a dedicated `_change_data` folder inside the Delta table directory.
You will use it in Task 3 to inspect exactly which rows were affected by each DML operation.

In [0]:
from pyspark.sql import functions as F

# Generate 2 million synthetic orders
# - 1 000 distinct product_ids  (PROD_0001 … PROD_1000)
# - 50 000 distinct customer_ids
# - 50 distinct store_ids
# This scale makes data-skipping differences visible in Task 6.
N = 2_000_000

df_synthetic = (
    spark.range(N)
    .select(
        F.concat(F.lit("ORD_"), F.lpad(F.col("id").cast("string"), 8, "0"))
         .alias("order_id"),
        (F.round(F.rand() * 49999).cast("long") + 1)
         .alias("customer_id"),
        F.concat(
            F.lit("PROD_"),
            F.lpad((F.round(F.rand() * 999).cast("long") + 1).cast("string"), 4, "0")
        ).alias("product_id"),
        F.concat(F.lit("STORE_"), (F.round(F.rand() * 49).cast("long") + 1).cast("string"))
         .alias("store_id"),
        F.from_unixtime(
            F.unix_timestamp(F.lit("2023-01-01")) + (F.rand() * 365 * 86400).cast("long")
        ).alias("order_datetime"),
        (F.round(F.rand() * 9).cast("int") + 1).alias("quantity"),
        F.round(F.rand() * 490 + 10, 2).alias("unit_price"),
        F.round(F.rand() * 30).cast("int").alias("discount_percent"),
        F.round(F.rand() * 490 + 10, 2).alias("total_amount"),
        F.element_at(
            F.array(F.lit("credit_card"), F.lit("debit_card"),
                    F.lit("paypal"), F.lit("bank_transfer"), F.lit("gift_card")),
            (F.round(F.rand() * 4).cast("int") + 1)
        ).alias("payment_method"),
    )
)

# Initial write: 200 files (high fragmentation)
spark.sql(f"DROP TABLE IF EXISTS {TABLE_BASE}")
df_synthetic.repartition(200).write.mode("overwrite").saveAsTable(TABLE_BASE)

# Simulate 20 micro-batch appends — 5 small files each = 100 more files
for i in range(20):
    (df_synthetic
     .sample(fraction=0.01, seed=i)  # ~20 000 rows per batch
     .repartition(5)
     .write.mode("append").saveAsTable(TABLE_BASE))

# Enable Change Data Feed
spark.sql(f"ALTER TABLE {TABLE_BASE} SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')")

total_rows  = spark.table(TABLE_BASE).count()
total_files = spark.sql(f"DESCRIBE DETAIL {TABLE_BASE}").first()["numFiles"]
print(f"Fragmented table ready. CDF enabled.")
print(f"  Total rows  : {total_rows:,}")
print(f"  Total files : {total_files}   <-- data is highly fragmented")

In [0]:
# Review schema and sample rows — you will need the column names in Task 3
df_base = spark.table(TABLE_BASE)
df_base.printSchema()
display(df_base.limit(3))

## Task 1: Inspect Table Metrics

Before making any changes, establish a baseline: how many files exist, how large is the table,
and what does the initial version history look like?

**What you need to do:**
1. Fill in `DESCRIBE DETAIL` to inspect storage metrics
2. Fill in `DESCRIBE HISTORY` to read the version log

**Guidance — Task 1**

**DESCRIBE DETAIL**
Returns one row of Delta metadata: `numFiles`, `sizeInBytes`, `clusteringColumns`, `partitionColumns`.
Use this before and after each optimization step to measure the impact.

**Example:**
```python
spark.sql("DESCRIBE DETAIL catalog.schema.table_name")
```

**DESCRIBE HISTORY**
Returns one row per Delta operation. Each row is a version — `version 0` is the first write.

**Example:**
```python
spark.sql("DESCRIBE HISTORY catalog.schema.table_name LIMIT 5")
```

| Command | Key columns to check |
|---|---|
| `DESCRIBE DETAIL` | `numFiles`, `sizeInBytes`, `clusteringColumns` |
| `DESCRIBE HISTORY` | `version`, `timestamp`, `operation`, `operationMetrics` |

**Things to think about**
- What is the ideal file size for a Delta table? (~128 MB – 1 GB per file)
- How many files would you expect in a well-maintained table of this size?

In [0]:
# TODO: Inspect table storage metrics with DESCRIBE DETAIL
# - Store the result in df_detail
# - Display: format, numFiles, sizeInBytes, clusteringColumns, partitionColumns
# - Save the file count into files_before for use in Task 2


In [0]:
# TODO: Read the 5 most recent versions from the transaction log
# - Store the result in df_history
# - Display: version, timestamp, operation, operationMetrics


In [0]:
# -- Validation --
detail       = df_detail.first()
files_before = detail["numFiles"]
assert detail["format"] == "delta", "Table must be Delta format"
print(f"Task 1 OK")
print(f"  Files      : {files_before}")
print(f"  Size       : {detail['sizeInBytes']:,} bytes")
print(f"  History rows returned : {df_history.count()}")

## Task 2: OPTIMIZE + VACUUM

With the baseline established, compact the small files and then permanently remove the obsolete ones.

**What you need to do:**
1. Run `OPTIMIZE` to merge small files into larger ones
2. Preview removable files with `VACUUM ... DRY RUN`
3. Run `VACUUM` to permanently remove obsolete files
4. Compare file count before and after

**Guidance — OPTIMIZE**

`OPTIMIZE` merges small Parquet files into files close to 1 GB (default target).

**Example:**
```sql
-- Compact all small files
OPTIMIZE catalog.schema.table_name

-- Compact AND co-locate rows with similar values (Z-ORDER)
OPTIMIZE catalog.schema.table_name ZORDER BY (product_id)
```

The operation is:
- **Safe** — queries continue to work during and after optimization
- **Idempotent** — running twice does nothing extra if no new data arrived
- **Transactional** — logged in `DESCRIBE HISTORY` as a new version

> Old files are kept for time travel until VACUUM removes them.

In [0]:
# TODO: Compact small files with OPTIMIZE
# After the command, run DESCRIBE DETAIL again and store the result in detail_after
# Print:
#   "Files BEFORE OPTIMIZE : {files_before}"
#   "Files AFTER OPTIMIZE  : {<new count>}"


**Guidance — VACUUM**

`VACUUM` permanently removes Parquet files no longer referenced by any Delta version older than the retention period.

**Example:**
```sql
-- Safe preview — shows which files WOULD be deleted (no actual deletion)
VACUUM catalog.schema.table_name RETAIN 168 HOURS DRY RUN

-- Execute — permanently deletes old files
VACUUM catalog.schema.table_name RETAIN 168 HOURS
```

The default retention is **168 hours (7 days)** — this protects time travel.
Setting `RETAIN 0 HOURS` removes all old files immediately.

> **Never run `RETAIN 0 HOURS` in production** — time travel will break for recent versions.

> **Note:** The updated file count in `DESCRIBE DETAIL` becomes accurate only after refreshing the Spark session (detach and re-attach the cluster, or restart the notebook kernel). Until then, Spark may report a stale file count from its metadata cache.

In [0]:
# Disable retention safety check — LAB ONLY, never do this in production!
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

# TODO: Preview which files would be deleted — use VACUUM with DRY RUN (RETAIN 0 HOURS)
# display() the result


In [0]:
# TODO: Execute VACUUM to permanently delete obsolete files (RETAIN 0 HOURS — lab only!)
# Print "VACUUM complete!" when done

# Re-enable safety check
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")


In [0]:
# -- Validation --
detail_v2         = spark.sql(f"DESCRIBE DETAIL {TABLE_BASE}").first()
rows_after_vacuum = spark.table(TABLE_BASE).count()
assert detail_v2["numFiles"] < files_before, "OPTIMIZE must have reduced file count"
assert rows_after_vacuum > 0, "Table must still contain data after VACUUM"
print(f"Task 2 OK")
print(f"  Files before OPTIMIZE : {files_before}")
print(f"  Files after OPTIMIZE  : {detail_v2['numFiles']}")
print(f"  Rows still present    : {rows_after_vacuum}")

## Task 3: INSERT / UPDATE / DELETE + Change Data Feed

> **Note:** The CDF demo (enabling, reading `_change_type`, querying changes) is in `BONUS_03_delta_advanced.ipynb`.

Delta Lake supports full ACID transactions, including row-level DML.
Each statement creates a new version in the transaction log — giving you a complete audit trail.
With CDF enabled, you can also read **exactly which rows changed** and how.

**What you need to do:**
1. Run the provided `INSERT` to add 10 test orders
2. Write an `UPDATE` to apply a 20 % discount to `ORD_TEST_001`
3. Write a `DELETE` to remove `ORD_TEST_008`, `ORD_TEST_009`, `ORD_TEST_010`
4. Inspect the version history with `DESCRIBE HISTORY`
5. Query the Change Data Feed to see every affected row across all three operations

**Guidance — DML on Delta Lake**

Delta supports full ACID transactions for all standard DML statements.
Each statement creates a **new version** in `DESCRIBE HISTORY`.

**Example — UPDATE:**
```sql
UPDATE catalog.schema.table_name
SET total_amount = total_amount * 0.9
WHERE order_id = 'ORD-001'
```

**Example — DELETE:**
```sql
DELETE FROM catalog.schema.table_name
WHERE order_id IN ('ORD-001', 'ORD-002')
```

> **Schema reminder:**
> `order_id`, `customer_id`, `product_id`, `store_id`, `order_datetime`,
> `quantity`, `unit_price`, `discount_percent`, `total_amount`, `payment_method`

---

**Change Data Feed (CDF)**

When CDF is enabled, every DML operation records the affected rows in a special `_change_data` folder.
You can read these changes with `table_changes()`.

**Example:**
```sql
-- Read all changes starting from version N
SELECT * FROM table_changes('catalog.schema.table_name', N)
```

The result contains your regular columns **plus** three metadata columns:

| Column | Description |
|---|---|
| `_change_type` | `insert`, `update_preimage`, `update_postimage`, `delete` |
| `_commit_version` | Delta version number of the operation |
| `_commit_timestamp` | Timestamp of the commit |

> For an UPDATE, CDF writes **two rows** per affected record: `update_preimage` (before) and `update_postimage` (after).
> This lets you see exactly what changed.

In [0]:
# INSERT — run as-is to add 3 new test orders (use these as reference for UPDATE and DELETE below)
spark.sql(f"""
    INSERT INTO {TABLE_BASE} (order_id, customer_id, product_id, store_id, order_datetime, quantity, unit_price, discount_percent, total_amount, payment_method) VALUES
    ('ORD_TEST_001', 'CUST000001', 'PROD000001', 'STORE001', '2025-01-15T10:00:00', 2, 49.99,   0,  99.98, 'Credit Card'),
    ('ORD_TEST_002', 'CUST000002', 'PROD000164', 'STORE002', '2025-01-15T11:00:00', 1, 206.74,  5, 196.40, 'Cash'),
    ('ORD_TEST_003', 'CUST000003', 'PROD000001', 'STORE001', '2025-01-16T09:30:00', 3, 49.99,  10, 134.97, 'Debit Card'),
    ('ORD_TEST_004', 'CUST000004', 'PROD000200', 'STORE003', '2025-01-16T14:00:00', 1,  89.99,  0,  89.99, 'Cash'),
    ('ORD_TEST_005', 'CUST000005', 'PROD000050', 'STORE005', '2025-01-17T08:15:00', 4,  24.99,  5,  94.96, 'Debit Card'),
    ('ORD_TEST_006', 'CUST000006', 'PROD000164', 'STORE010', '2025-01-17T12:30:00', 2, 206.74,  0, 413.48, 'Credit Card'),
    ('ORD_TEST_007', 'CUST000007', 'PROD000001', 'STORE002', '2025-01-18T09:00:00', 1,  49.99, 15,  42.49, 'Cash'),
    ('ORD_TEST_008', 'CUST000008', 'PROD000075', 'STORE007', '2025-01-18T16:45:00', 2, 119.99,  0, 239.98, 'Debit Card'),
    ('ORD_TEST_009', 'CUST000009', 'PROD000200', 'STORE001', '2025-01-19T10:10:00', 3,  89.99, 10, 242.97, 'Credit Card'),
    ('ORD_TEST_010', 'CUST000010', 'PROD000050', 'STORE003', '2025-01-19T13:55:00', 5,  24.99,  0, 124.95, 'Cash')
""")
print("Inserted 10 new test orders")

In [0]:
# TODO: Apply a 20 % discount to ORD_TEST_001
# UPDATE total_amount (multiply by 0.8) WHERE order_id matches
# Print "UPDATE complete" when done


In [0]:
# TODO: Delete ORD_TEST_008, ORD_TEST_009, and ORD_TEST_010 in a single DELETE statement
# Use the IN operator to match all three order_ids at once
# Print "DELETE complete" when done


In [0]:
# TODO: Read the transaction log — check that INSERT, UPDATE, DELETE appear as separate versions
# - Limit to 8 versions
# - Store the result in df_hist_dml
# - Display: version, timestamp, operation, operationMetrics


In [0]:
# TODO: Query the Change Data Feed for all row-level changes from the INSERT onwards
#
# Step 1 — find the version number of the INSERT operation in df_hist_dml
#           (look for operation == "WRITE" or "INSERT")
#           store it in: insert_version
#
# Step 2 — query table_changes() starting from that version
#           store the result in: df_cdf
#
# Step 3 — display order_id, total_amount, _change_type, _commit_version, _commit_timestamp
#           ordered by _commit_version, order_id


In [0]:
# -- Validation --
ops = [r["operation"] for r in df_hist_dml.collect()]
assert any(o in ops for o in ("WRITE", "INSERT")), "Expected an INSERT/WRITE in history"
assert "UPDATE" in ops, "Expected UPDATE in history"
assert "DELETE" in ops, "Expected DELETE in history"
rows_now = spark.table(TABLE_BASE).count()
print(f"Task 3 OK")
print(f"  DML operations found : {[o for o in ops if o in ('WRITE','INSERT','UPDATE','DELETE')]}")
print(f"  Rows after DELETE    : {rows_now}")

## Task 4: Time Travel

Delta Lake retains all historical versions of the table.
You can query any past version and restore the table to a previous state — without a separate backup system.

**What you need to do:**
1. Read the table as it existed at version 1 (after OPTIMIZE, before DML)
2. Compare row counts between version 1 and the current version
3. Restore the table to version 1 using `RESTORE TABLE`
4. Confirm the restoration by checking the final row count

**Guidance — Time Travel**

Query a historical version:
```sql
SELECT * FROM table VERSION AS OF 1
SELECT * FROM table TIMESTAMP AS OF '2025-01-01'
```

Or with PySpark:
```python
spark.read.format("delta").option("versionAsOf", 1).table("table_name")
```

`RESTORE TABLE table TO VERSION AS OF N` rolls the table back to version N.
This creates a **new version** — the full history is preserved, not truncated.

**When to use time travel**
- Accidental DELETE or UPDATE in production
- Compliance audits requiring point-in-time access
- Comparing data quality between pipeline runs

In [0]:
# Step 1: Show the full transaction history
display(spark.sql(f"DESCRIBE HISTORY {TABLE_BASE}"))

# TODO: Look at the history above and identify the version number for the OPTIMIZE operation
# Assign it to _opt_v (it should be version 1 if OPTIMIZE ran second)
_opt_v = ...  # Replace ... with the correct version number

In [0]:
df_v1      = spark.read.format("delta").option("versionAsOf", _opt_v).table(TABLE_BASE)
count_v1   = df_v1.count()
count_curr = spark.table(TABLE_BASE).count()

print(f"Version {_opt_v} row count  : {count_v1}")
print(f"Current row count    : {count_curr}")


display(df_v1)

In [0]:
# TODO: Restore the table to the OPTIMIZE version to get the compacted file layout
# Use RESTORE TABLE ... TO VERSION AS OF <version>
spark.sql(f"""
    -- Your RESTORE TABLE statement here
""")
print("RESTORE complete!")

In [0]:
# Compare counts after restore
count_restored = spark.table(TABLE_BASE).count()
latest_op      = spark.sql(f"DESCRIBE HISTORY {TABLE_BASE} LIMIT 1").first()

print(f"Rows before restore (current) : {count_curr}")
print(f"Rows after RESTORE            : {count_restored}")
print(f"New version created           : {latest_op['version']}")
print(f"Operation logged              : {latest_op['operation']}")

In [0]:
# -- Validation --
assert count_restored == count_v1, f"Restored count {count_restored} should match v1 ({count_v1})"
assert latest_op["operation"] == "RESTORE", "Last operation should be RESTORE"
total_versions = spark.sql(f"DESCRIBE HISTORY {TABLE_BASE}").count()
print(f"Task 4 OK")
print(f"  Table restored to version 1 ({count_restored} rows)")
print(f"  Full history preserved — total versions: {total_versions}")

## Task 5: Deep Clone

> **Note:** The CLONE demo (Shallow vs Deep Clone comparison) is in `BONUS_03_delta_advanced.ipynb`.

Before applying different optimization strategies, create two independent copies of the table.
Deep Clone copies both the data files and the transaction log — the clone is fully independent of the source.

**What you need to do:**
1. Create `orders_lab_zorder` as a deep clone of the base table
2. Create `orders_lab_liquid` as a deep clone of the base table
3. Verify both clones contain the correct row count

**Guidance — DEEP CLONE**

```sql
CREATE OR REPLACE TABLE target
DEEP CLONE source
```

`DEEP CLONE` physically copies all data files and the Delta transaction log.
The clone is **completely independent** — changes to the source do not affect the clone and vice versa.

Contrast with `SHALLOW CLONE`: copies only the transaction log (metadata), data files stay in the source location.
Shallow clones are faster but break if the source runs `VACUUM`.

**Use DEEP CLONE when:**
- You need a sandbox for testing optimization strategies without risk
- You want a full backup including complete time travel history
- You need to test schema changes in isolation

In [0]:
# TODO: Deep clone TABLE_BASE into TABLE_ZORDER
# Syntax: CREATE OR REPLACE TABLE <target> DEEP CLONE <source>
spark.sql(f"""
    -- Your DEEP CLONE statement here
""")
print(f"Z-ORDER clone created: {spark.table(TABLE_ZORDER).count()} rows")

In [0]:
# TODO: Deep clone TABLE_BASE into TABLE_LIQUID (same syntax as above)
spark.sql(f"""
    -- Your DEEP CLONE statement here
""")
print(f"Liquid clone created: {spark.table(TABLE_LIQUID).count()} rows")

In [0]:
# -- Validation --
base_count = spark.table(TABLE_BASE).count()
for table, label in [(TABLE_ZORDER, "Z-ORDER"), (TABLE_LIQUID, "Liquid")]:
    count = spark.table(table).count()
    assert count == base_count, f"{label} clone row count {count} != base {base_count}"
    print(f"  {label} clone OK: {count} rows")
print(f"Task 5 OK: Both clones are ready for independent optimization.")

## Task 6: Z-ORDER vs Liquid Clustering vs Baseline

With three independent table copies, apply different optimization strategies and measure
their impact on queries that filter by `product_id`.

**What you need to do:**
1. Apply `OPTIMIZE ZORDER BY (product_id)` on the Z-ORDER table (provided)
2. Enable Liquid Clustering on the Liquid table, then run `OPTIMIZE` (fill in the blank)
3. Run the same filtered query on all three tables and compare timing
4. Use `EXPLAIN` to inspect each physical plan
5. Open **Spark UI → SQL tab** and compare `files read` vs `files pruned` across all three tables

> **Before running benchmark queries:** Open the Spark UI (cluster button in Databricks) and navigate to the SQL tab.

**Guidance — Three Optimization Strategies**

The table has **~2.4 million rows across ~300 files**. Because `product_id` values are randomly
distributed, each product appears in nearly every file. A filter like `WHERE product_id = 'PROD_0042'`
would force Spark to open all 300 files even though only ~2 000 rows match.

Optimization solves this by **co-locating similar values in the same files** — so Spark can skip
the files that cannot contain the target product.

| Strategy | Command | Data skipping mechanism | Best for |
|---|---|---|---|
| Baseline | — | Min/max stats per file | Small or write-heavy tables |
| Z-ORDER | `OPTIMIZE ... ZORDER BY (col)` | Z-curve co-location | Stable query patterns, 1-2 columns |
| Liquid Clustering | `CLUSTER BY (col)` + `OPTIMIZE` | Hilbert curve, incremental | Evolving patterns, large tables |

**Example — Z-ORDER:**
```sql
-- Co-locate rows with similar product_id values in the same files
OPTIMIZE catalog.schema.table_name ZORDER BY (product_id)
```

**Example — Liquid Clustering:**
```sql
-- Step 1: declare the clustering key (written to table metadata)
ALTER TABLE catalog.schema.table_name CLUSTER BY (product_id)

-- Step 2: physically reorganize the data
OPTIMIZE catalog.schema.table_name
```

**How data skipping works**
The Delta transaction log stores min/max values per file per column.
When you filter `WHERE product_id = 'X'`, Spark skips files whose min/max range does not overlap `'X'`.
Both Z-ORDER and Liquid Clustering improve this by co-locating similar values in fewer files.
After running Task 6, the Benchmark cell will show you exactly how many files each strategy reads.

In [0]:
# TODO: Run OPTIMIZE with ZORDER BY on TABLE_ZORDER, clustering by product_id
# Syntax: OPTIMIZE <table> ZORDER BY (col)
spark.sql(f"...")
print("Z-ORDER optimization applied.")

detail_zorder = spark.sql(f"DESCRIBE DETAIL {TABLE_ZORDER}").first()
print(f"  Files after OPTIMIZE: {detail_zorder['numFiles']}")

In [0]:
# TODO Step 1: Enable Liquid Clustering on TABLE_LIQUID by product_id
# Syntax: ALTER TABLE <table> CLUSTER BY (col)
spark.sql(f"""
    -- Your ALTER TABLE statement here
""")

# TODO Step 2: Run OPTIMIZE to physically apply the clustering
spark.sql(f"OPTIMIZE {TABLE_LIQUID}")

print("Liquid Clustering applied.")
detail_liquid = spark.sql(f"DESCRIBE DETAIL {TABLE_LIQUID}").first()
print(f"  Files after OPTIMIZE  : {detail_liquid['numFiles']}")
print(f"  Clustering columns    : {detail_liquid['clusteringColumns']}")

## Benchmark: Query Performance Comparison

Now that all three tables are optimized (or not), run the same point query against each one.
The benchmark measures:

- **Files in table** — how many Parquet files Delta has to consider
- **Query time** — wall-clock seconds for `WHERE product_id = 'PROD_0042'`

A lower file count means Delta can skip more files → faster scans.
This is what data-skipping optimization gives you.

In [ ]:
import time

BENCHMARK_PRODUCT = "PROD_0042"

def time_query(table):
    """Run a point-query for one product and return (elapsed_seconds, row_count)."""
    t0 = time.time()
    result = spark.sql(f"""
        SELECT product_id, COUNT(*) AS order_count, ROUND(SUM(total_amount), 2) AS total_revenue
        FROM {table}
        WHERE product_id = '{BENCHMARK_PRODUCT}'
        GROUP BY product_id
    """).collect()
    elapsed = time.time() - t0
    rows = result[0]["order_count"] if result else 0
    return elapsed, rows

print(f"Benchmark: WHERE product_id = '{BENCHMARK_PRODUCT}'")
print(f"{'Strategy':<30} {'Files':>7} {'Time (s)':>10} {'Rows matched':>14}")
print("-" * 65)

for table, label in [
    (TABLE_BASE,   "Baseline (unoptimized)"),
    (TABLE_ZORDER, "Z-ORDER by product_id"),
    (TABLE_LIQUID, "Liquid Clustering by product_id"),
]:
    n_files = spark.sql(f"DESCRIBE DETAIL {table}").first()["numFiles"]
    t, rows = time_query(table)
    print(f"{label:<30} {n_files:>7} {t:>10.2f} {rows:>14,}")

print()
print("Key insight: Z-ORDER and Liquid Clustering reduce files scanned")
print("for a point query, even though the total row count is the same.")

## Cleanup

In [0]:
for table in [TABLE_BASE, TABLE_ZORDER, TABLE_LIQUID]:
    spark.sql(f"DROP TABLE IF EXISTS {table}")
    print(f"Dropped: {table}")

spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "true")
print("All lab tables cleaned up.")

## Lab Complete

You have practiced the full Delta Lake optimization lifecycle:

| Task | Skill |
|---|---|
| 1 — Inspect | `DESCRIBE DETAIL`, `DESCRIBE HISTORY` — establishing a baseline |
| 2 — Optimize | `OPTIMIZE` + `VACUUM` — file compaction and storage reclaim |
| 3 — DML + CDF | `INSERT`, `UPDATE`, `DELETE` — ACID transactions + `table_changes` row-level audit |
| 4 — Time Travel | `VERSION AS OF`, `RESTORE TABLE` — point-in-time recovery |
| 5 — Clone | `DEEP CLONE` — isolated copies for safe experimentation |
| 6 — Compare | Z-ORDER vs Liquid Clustering vs baseline — data-skipping benchmark |

> **Key takeaway:** Liquid Clustering is the modern replacement for both partitioning and Z-ORDER.
> Use `ALTER TABLE ... CLUSTER BY (new_col)` to change clustering columns without rewriting all data.

> **Next:** [02 — Security & Governance Workshop](02_security_governance_workshop.ipynb)

← [04 — Delta Optimization](../demo/04_delta_optimization.ipynb) | **[ README](../../../README.md)** | [05 — Incremental Ingestion →](../demo/05_incremental_ingestion.ipynb)